In [ ]:
import csv
import json
import requests
import numpy as np
import pandas as pd

# ==========================================
# Task 1.1 — Read the file the foundations way
# ==========================================
def parse_row(row):
    parsed = dict(row)
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed

accepted, rejected = [], []
with open("data/appointments.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    print("Header:", reader.fieldnames)
    row_count = 0
    for row in reader:
        row_count += 1
        try:
            accepted.append(parse_row(row))
        except ValueError:
            rejected.append(row)

print(f"Total rows: {row_count} | Accepted: {len(accepted)} | Rejected: {len(rejected)}")

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)


# ==========================================
# Task 1.2 — Inspect and clean, pass 1
# ==========================================
df = pd.read_csv("data/appointments.csv")
print("Raw shape:", df.shape)
print(df.info())
print("\nNull counts:\n", df.isnull().sum())
print("\nExact duplicates:", df.duplicated().sum())
print("Raw booking_channel distinct values:", df["booking_channel"].nunique())

# (a) Drop exact duplicates
df = df.drop_duplicates().copy()
print("Row count after (a) drop_duplicates:", len(df))

# (b) Clean booking_channel
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("Cleaned booking_channel values:", df["booking_channel"].unique())

# (c) Convert dates
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])

# (d) Drop missing appt_date or status
df = df.dropna(subset=["appt_date", "status"]).copy()
print("Row count after (d) dropna:", len(df))


# ==========================================
# Task 1.3 — Clean, pass 2: validity, gaps and outliers
# ==========================================
# (a) Invalid ages -> NaN
invalid_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages set to NaN:", invalid_age.sum())
df.loc[invalid_age, "age"] = np.nan

# (b) Fill missing age and distance_km with medians
med_age = df["age"].median()
med_dist = df["distance_km"].median()
print(f"Median age: {med_age} | Median distance_km: {med_dist}")

df["age"] = df["age"].fillna(med_age).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(med_dist)

# (c) IQR capping for distance_km and fee_inr
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    capped_count = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = np.clip(df[col], lower, upper)
    print(f"{col} -> Q1: {q1}, Q3: {q3}, Upper fence: {upper}, Capped: {capped_count}")

# (d) Round
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)
print("Max fee_inr after capping & rounding:", df["fee_inr"].max())


# ==========================================
# Task 1.4 — Features and a NumPy summary
# ==========================================
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

print(f"lead_days range: {df['lead_days'].min()} to {df['lead_days'].max()}")
print(f"No-shows: {df['no_show'].sum()} / {len(df)} ({df['no_show'].mean()*100:.1f}%)")

X_np = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
col_means = np.mean(X_np, axis=0)
print("Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

mask_lead = X_np[:, 3] > 14
print(f"lead_days > 14: {mask_lead.sum()} rows ({mask_lead.mean()*100:.1f}%)")

dist_vals = X_np[:, 1]
z_dist = (dist_vals - np.mean(dist_vals)) / np.std(dist_vals)
print("Rows with |z| > 2 for distance_km:", (np.abs(z_dist) > 2).sum())


# ==========================================
# Task 1.5 — Enrich the clinics from a live API
# ==========================================
clinics = pd.read_csv("data/clinics.csv")
temps = []

for _, row in clinics.iterrows():
    temp_val = None
    reason = ""
    for attempt in range(2):
        try:
            resp = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": row["latitude"],
                    "longitude": row["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1,
                },
                timeout=10,
            )
            if resp.status_code == 200:
                payload = resp.json()
                if "hourly" in payload and "temperature_2m" in payload["hourly"]:
                    temp_val = payload["hourly"]["temperature_2m"][0]
                    break
                else:
                    reason = "Missing hourly/temperature_2m in response"
            else:
                reason = f"HTTP {resp.status_code}"
        except requests.exceptions.RequestException as e:
            reason = str(e)
    if temp_val is None:
        print(f"Failed for {row['clinic_name']}: {reason}")
    temps.append(temp_val)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)
print(clinics[["clinic_id", "clinic_name", "temp_c"]])


# ==========================================
# Task 1.6 — Export clean table & write load_data.sql
# ==========================================
cols_16 = [
    "appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
    "age", "gender", "distance_km", "fee_inr", "booking_channel",
    "visit_type", "reminder_sent", "prior_no_shows", "status", "lead_days", "no_show"
]

df = df.sort_values("appointment_id").reset_index(drop=True)
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")
df = df[cols_16]

print("Unique appointment_id:", df["appointment_id"].nunique())
print("Total missing values across 16 columns:", df.isnull().sum().sum())
df.to_csv("appointments_clean.csv", index=False)

with open("load_data.sql", "w", encoding="utf-8") as f:
    # Clinics first (parent table for foreign key)
    for _, r in clinics.iterrows():
        temp_sql = "NULL" if pd.isna(r["temp_c"]) else f"{r['temp_c']}"
        f.write(
            f"INSERT INTO clinics VALUES ('{r['clinic_id']}','{r['clinic_name']}',"
            f"'{r['city']}','{r['tier']}',{r['latitude']},{r['longitude']},{temp_sql});\n"
        )
    # Appointments second
    for _, r in df.iterrows():
        f.write(
            f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}',"
            f"'{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{r['age']},"
            f"'{r['gender']}',{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}',"
            f"'{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},"
            f"'{r['status']}',{r['lead_days']},{r['no_show']});\n"
        )

print("load_data.sql written successfully.")